# 05 - Dashboard & Multi-Location Demo

Demonstrates the Panel dashboard embedding, multi-location comparison,
Datashader global maps, and Lumen pipeline integration.

In [ ]:
import panel as pn
import holoviews as hv
import numpy as np
hv.extension('bokeh')
pn.extension()

## Multi-Location Comparison

In [ ]:
from solar_intelligence.solar_analysis import MultiLocationComparator
from solar_intelligence.visualization import SolarVisualizer

locations = {
    'New Delhi': (28.6, 77.2),
    'London': (51.5, -0.1),
    'Cairo': (30.0, 31.2),
    'Tokyo': (35.7, 139.7),
    'Sydney': (-33.9, 151.2),
}

comp = MultiLocationComparator(locations=locations)
comp.load_data(start_year=2023, end_year=2023)
viz = SolarVisualizer(width=700, height=400)

In [ ]:
ranking = comp.ranking()
print(ranking[['rank', 'location', 'GHI', 'annual_kwh_m2']].to_string(index=False))

In [ ]:
viz.multi_location_bar(comp.compare_ghi())

In [ ]:
viz.multi_location_monthly(comp.compare_monthly())

## Datashader Global Solar Map

In [ ]:
from solar_intelligence.data_loader import generate_global_solar_grid

global_ds = generate_global_solar_grid(resolution=0.5)
print(f"Grid: {global_ds['GHI'].shape[0]} x {global_ds['GHI'].shape[1]} = {global_ds['GHI'].size:,} points")
viz.datashader_global_map(global_ds)

## Dynamic Rasterization

In [ ]:
viz.dynamic_rasterized_map(global_ds)

## Lumen Pipeline

In [ ]:
from solar_intelligence.ui.lumen_app import SolarDataSource, SolarEnergyTransform

source = SolarDataSource(latitude=28.6, longitude=77.2, use_synthetic=True, start_year=2023, end_year=2023)
df = source.get('daily_solar')

transform = SolarEnergyTransform(panel_efficiency=0.20, total_area=34.0)
df_with_energy = transform.apply(df)
print(f"Daily energy range: {df_with_energy['energy_kwh'].min():.1f} - {df_with_energy['energy_kwh'].max():.1f} kWh")
print(f"Annual total: {df_with_energy['energy_kwh'].sum():,.0f} kWh")

## Embed Dashboard (Panel serve)

To run the full interactive dashboard:
```bash
panel serve src/solar_intelligence/ui/panel_dashboard.py --show
```

In [ ]:
from solar_intelligence.ui.panel_dashboard import SolarDashboard

dashboard = SolarDashboard()
# In a notebook, call dashboard.view() to embed inline
print('Dashboard created. Run `panel serve` for full interactivity.')